# Import

In [ ]:
from datetime import datetime, timedelta

import inflection
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots


# Carregando Dados

In [ ]:
df_sales_raw = pd.read_csv( r"C:\Users\Admin\.cache\kagglehub\competitions\rossmann-store-sales\train.csv", low_memory=False )
df_store_raw = pd.read_csv( r"C:\Users\Admin\.cache\kagglehub\competitions\rossmann-store-sales\store.csv", low_memory=False )

df_raw = pd.merge (df_sales_raw, df_store_raw, how="left", on="Store")


# 1 Passo 1 - Entendimento dos Dados

In [ ]:
df1 = df_raw.copy()

## 1.1 Renomeando As Colunas

In [ ]:
cols_old = ['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 
            'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 
            'Promo2SinceYear', 'PromoInterval'
            ]

snakecase = lambda x: inflection.underscore( x )

cols_new = list( map ( snakecase, cols_old ))

df1.columns = cols_new

## 1.2 Dimensão Dos Dados

In [ ]:
print( f"Numero de linhas: {df1.shape[0]}" )
print( f"Numero de colunas: {df1.shape[1]}" )

## 1.3 Tipo De Dados

In [ ]:
df1['date'] = pd.to_datetime(df1['date'])
df1.dtypes

## 1.4 Analisar os NA

In [ ]:
df1.isna().sum()

## 1.5 Filtrando os NA

In [ ]:
month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

df1['competition_distance'] = df1['competition_distance'].fillna(200000)

df1['competition_open_since_month'] = df1['competition_open_since_month'].fillna(df1['date'].dt.month)
df1['competition_open_since_year'] = df1['competition_open_since_year'].fillna(df1['date'].dt.year)

df1['promo2_since_week'] = df1['promo2_since_week'].fillna(df1['date'].dt.isocalendar().week)
df1['promo2_since_year'] = df1['promo2_since_year'].fillna(df1['date'].dt.year)

df1['promo_interval'] = df1['promo_interval'].fillna(0)
df1['month_map'] = df1['date'].dt.month.map(month_map)

df1['is_promo'] = df1[['promo_interval', 'month_map']].apply(
    lambda x: 0 if x['promo_interval'] == 0 else (1 if x['month_map'] in str(x['promo_interval']).split(',') else 0),
    axis=1
)

## 1.6 Mudando O Tipo Dos Dados

In [ ]:
df1['competition_open_since_month'] = df1['competition_open_since_month'].astype(int)
df1['competition_open_since_year'] = df1['competition_open_since_year'].astype( int )
df1['promo2_since_week'] = df1['promo2_since_week'].astype(int)
df1['promo2_since_year'] = df1['promo2_since_year'].astype(int)

## 1.7 Descrição Estatistica

In [ ]:
num_attributes = df1.select_dtypes(include=['int64', 'float64'])
cat_attributes = df1.select_dtypes(exclude=['int64', 'float64', 'datetime64'])

### 1.7.1 Atributos Numericos

In [ ]:
# Tendencias centrais - Media e Mediana
m1 = pd.DataFrame(num_attributes.apply(np.mean)).T
m2 = pd.DataFrame(num_attributes.apply(np.median)).T

# Dispersão - std, min, max, range, skew e kurtosis
d1 = pd.DataFrame(num_attributes.apply(np.std)).T
d2 = pd.DataFrame(num_attributes.apply(np.min)).T
d3 = pd.DataFrame(num_attributes.apply(np.max)).T
d4 = pd.DataFrame(num_attributes.apply(lambda x: x.max() - x.min())).T
d5 = pd.DataFrame(num_attributes.apply(lambda x: x.skew())).T
d6 = pd.DataFrame(num_attributes.apply(lambda x: x.kurtosis())).T

# Concatena
m = pd.concat([d2, d3, d4, m1, m2, d1, d5, d6]).T
m.columns = ['min', 'max', 'range', 'mean', 'median', 'std', 'skew', 'kurtosis']
m

### 1.7.2 Atributos Categoricos

In [ ]:
cat_attributes.apply(lambda x: x.unique().shape[0])

In [ ]:
aux1 = df1[(df1['state_holiday'] != '0') & (df1['sales'] > 0)]

fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=('State Holiday vs Sales', 'Store Type vs Sales', 'Assortment vs Sales')
)

fig1 = px.box(aux1, x='state_holiday', y='sales', color='state_holiday')
fig2 = px.box(aux1, x='store_type', y='sales', color='store_type')
fig3 = px.box(aux1, x='assortment', y='sales', color='assortment')

for trace in fig1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in fig3.data:
    fig.add_trace(trace, row=1, col=3)

fig.update_layout(
    height=600,             # Altura do gráfico em pixels
    autosize=True,          # Preenche a largura total disponível
    template='plotly_white', # Tema limpo com alto contraste
    showlegend=False,       # Opcional: oculta a legenda lateral para dar mais espaço
    title_text='Análise de Vendas por Atributos Categóricos'
)

fig.show()

# 2 Passo 2 - Preparação dos Dados

In [ ]:
df2 = df1.copy()

## 2.1 Criação Das Hipoteses

### 2.1.1 Hipoteses Loja

**1.** Lojas com maior quadro de funcionarios deveriam vender mais.

**2.** Lojas com maior estoque deveriam vender mais.

**3.** Lojas com maior porte deveriam vender mais.

**4.** Lojas com maior porte deveriam vender menos

**5.** Lojas com maior sortimento deveriam vender mais.

**6.** Lojas com competidores a mais tempo deveriam vender mais.

### 2.1.2 Hipoteses Produtos

**1.** Lojas que investem mais em marketing deveriam vender mais.

**2.** Lojas que expoe mais os produtos nas vitrines deveriam vender mais.

**3.** Lojas que tem preços menores nos produtos deveriam vender mais.

**4.** Lojas com promoções mais agressivas (descontos maiores), deveriam vender mais.

**5.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**6.** Lojas com mais dias de promoções deveriam vender mais.

**7.** Lojas com mais promoções consecutivas deveriam vender mais.

### 2.1.3 Hipoteses Tempo

**1.** Lojas que tem mais feriados deveriam vender menos.

**2.** Lojas que abrem nos primeiros 6 meses deveriam vender mais.

**3.** Lojas que abrem nos finais de semana deveriam vender mais.

**4.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**5.** Lojas deveriam vender menos aos finais de semana.

**6.** Lojas deveriam vender durante os feriados escolares.

## 2.2 Lista Final de Hipóteses

**1.** Lojas com maior sortimentos deveriam vender mais.

**2.** Lojas com competidores mais proximos deveriam vender menos.

**3.** Lojas com competidores à mais deveriam vender mais.

**4.** Lojas com promoções ativas por mais tempo deveriam vender mais.

**5.** Lojas com mais dias de promoção deveriam vender mais.

**6.** Lojas com mais promoções consecutivas deveriam vender mais.

**7.** Lojas abertas durante o feriado de Natal deveriam vender mais.

**8.** Lojas deveriam vender mais ao longo dos anos.

**9.** Lojas deveriam vender mais no segundo semestre do ano.

**10.** Lojas deveriam vender mais depois do dia 10 de cada mês.

**11.** Lojas deveriam vender menos aos finais de semana.

**12.** Lojas deveriam vender menos durante os feriados escolares.

## 2.3 Engenharia de Recursos

In [ ]:
# Colunas de tempo
df2['year'] = df2['date'].dt.year
df2['month'] = df2['date'].dt.month
df2['day'] = df2['date'].dt.day
df2['week_of_year'] = df2['date'].dt.isocalendar().week

df2['year_week'] = df2['date'].dt.strftime( '%Y-%W' )

# Colunas de Competição por tempo
year = df2['competition_open_since_year'].astype(int).astype(str)
month = df2['competition_open_since_month'].astype(int).astype(str)

df2['competition_since'] = pd.to_datetime(year + '-' + month + '-01', errors='coerce')
df2['competition_time_month'] = ((df2['date'] - df2['competition_since']).dt.days / 30).astype(int)

# Colunas de Promoção
df2['date'] = df2['date'].dt.tz_localize(None)
promo_date_str = df2['promo2_since_year'].astype(str) + '-' + df2['promo2_since_week'].astype(str) + '-1'

df2['promo2_since'] = pd.to_datetime(promo_date_str, format='%Y-%W-%w') - pd.Timedelta(days=7)
df2['promo_time_week'] = ((df2['date'] - df2['promo2_since']).dt.days / 7).astype(int)

#Sortimento
df2['assortment'] = df2['assortment'].apply(lambda x: 'basic' if x == 'a' else 'extra' if x == 'b' else 'extended')

#Feriados nacionais
# Mapeamento dos feriados
HOLIDAY_MAP = {
    'a': 'public_holiday',
    'b': 'easter_holiday',
    'c': 'christmas'
}
df2['state_holiday'] = df2['state_holiday'].map(HOLIDAY_MAP).fillna('regular_day')
